# API Cosm'éthique

Notebook du site FastAPI et de son modèle de recommandation. Les imports, paramètres, classes et variables globales sont regroupés au début. Exécutez les cellules dans l'ordre ; les deux dernières démarrent et arrêtent le site local.

Pour installer les dépendances dans l'environnement du projet, utilisez `%pip install -r requirements.txt`.


# 1 - Définition des variables et import

## A - Import

In [1]:
from pathlib import Path
from typing import Dict, List, Optional, Set
import threading
import time

import numpy as np
import pandas as pd
import uvicorn
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import FileResponse
from fastapi.staticfiles import StaticFiles
from pydantic import BaseModel, Field
from sklearn.feature_extraction import DictVectorizer
from sklearn.metrics.pairwise import cosine_similarity

## B - Définition des chemins de fichiers

In [2]:
BASE_DIR = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
if not (BASE_DIR / "data" / "products.csv").is_file():
    BASE_DIR = next((path for path in (Path.cwd(), *Path.cwd().parents) if (path / "data" / "products.csv").is_file()), BASE_DIR)

DATA_DIR = BASE_DIR / "data"
STATIC_DIR = BASE_DIR / "static"
CSV_PATH = DATA_DIR / "products.csv"
PRODUCT_INGREDIENTS_PATH = DATA_DIR / "product_ingredients.csv"
PRODUCT_IMAGES_DIR = DATA_DIR / "images"

METADATA_FEATURES = ("category", "subcategory", "product_type")

## C - Classes 

In [ ]:
# Classe qui définit le format de la requête de recommandation.
class RecommendationRequest(BaseModel):
    selected_ids: List[int] = Field(..., min_items=1, max_items=5, description="De 1 à 5 identifiants de produits")
    excluded_ingredients: List[str] = Field(default_factory=list, description="Ingrédients à exclure")
    top_n: int = Field(default=3, ge=1, le=20)

# Classe qui définit le format de la réponse de recommandation.
class ProductResponse(BaseModel):
    id: int
    brand: str = ""
    name: str = ""
    image_name: Optional[str] = None
    category: str = ""
    subcategory: str = ""
    size: Optional[str] = None
    benefits: Optional[str] = None
    origin: str = ""
    product_type: str = ""
    ingredients_raw: str = ""
    contains_fragrance: bool = False
    contains_drying_alcohol: bool = False
    contains_parabens: bool = False
    contains_sulfates: bool = False
    contains_silicones: bool = False
    similarity_score: Optional[float] = None

C:\Users\User\AppData\Local\Temp\ipykernel_20976\574462816.py:3: PydanticDeprecatedSince20: `min_items` is deprecated and will be removed, use `min_length` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.14/migration/
  selected_ids: List[int] = Field(..., min_items=1, max_items=5, description="De 1 à 5 identifiants de produits")
C:\Users\User\AppData\Local\Temp\ipykernel_20976\574462816.py:3: PydanticDeprecatedSince20: `max_items` is deprecated and will be removed, use `max_length` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.14/migration/
  selected_ids: List[int] = Field(..., min_items=1, max_items=5, description="De 1 à 5 identifiants de produits")


# II - Construction de l'API utilisée sur le site

In [ ]:
# Initialisation de l'application FastAPI et configuration du middleware CORS
app = FastAPI(title="Cosm'éthique", description="Site de recommandation de produits cosmétiques", version="1.1.0")
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_credentials=True, allow_methods=["*"], allow_headers=["*"])
app.mount("/images", StaticFiles(directory=PRODUCT_IMAGES_DIR), name="product-images")


In [ ]:
# Charge les données et préparer la matrice de caractéristiques
df = ingredient_data = pd.DataFrame()
feature_vectorizer = DictVectorizer()
feature_matrix = cosine_sim = None
server = server_thread = None

In [ ]:
# Fonction pour charger les données à partir des fichiers CSV et préparer les df
def load_and_prepare_data(products_path: str, product_ingredients_path: str) -> tuple[pd.DataFrame, pd.DataFrame]:
    if not Path(products_path).is_file():
        raise FileNotFoundError(f"Fichier CSV des produits introuvable : {products_path}")
    if not Path(product_ingredients_path).is_file():
        raise FileNotFoundError(f"Fichier CSV des ingrédients introuvable : {product_ingredients_path}")

    products = pd.read_csv(products_path, keep_default_na=False)
    ingredients = pd.read_csv(product_ingredients_path, usecols=["product_id", "ingredient_id", "label_name"], dtype=str, keep_default_na=False)
    text_columns = ("brand", "name", "category", "subcategory", "ingredients_raw", "origin", "product_type", "benefits", "directions", "warnings")
    bool_columns = ("contains_fragrance", "contains_drying_alcohol", "contains_parabens", "contains_sulfates", "contains_silicones")

    for column in ("id", *text_columns, *bool_columns):
        if column not in products.columns:
            products[column] = ""
    products[list(text_columns)] = products[list(text_columns)].fillna("").astype(str)
    products["id"] = pd.to_numeric(products["id"], errors="raise").astype(int)
    for column in bool_columns:
        products[column] = products[column].map(lambda value: value if isinstance(value, bool) else str(value).strip().lower() == "true")

    ingredients["product_id"] = pd.to_numeric(ingredients["product_id"], errors="raise").astype(int)
    ingredients["ingredient_id"] = ingredients["ingredient_id"].str.strip()
    return products, ingredients[ingredients["ingredient_id"] != ""]

In [ ]:
# Fonction pour construire les lignes de caractéristiques à partir des produits et des ingrédients
def build_feature_rows(products: pd.DataFrame, ingredients: pd.DataFrame) -> List[Dict[str, float]]:
    by_product = ingredients.groupby("product_id")["ingredient_id"].agg(set).to_dict()
    rows = []
    for product in products.itertuples(index=False):
        features = {f"ingredient:{ingredient_id}": 1.0 for ingredient_id in by_product.get(product.id, set())}
        features.update({f"{column}:{value}": 1.0 for column in METADATA_FEATURES if (value := str(getattr(product, column, "")).strip().casefold())})
        rows.append(features)
    return rows


In [ ]:
# Chemin pour la page d'accueil qui sert le fichier index.html
@app.get("/", include_in_schema=False)
def home():
    return FileResponse(STATIC_DIR / "index.html")

In [ ]:
#Charge les données et prépare la matrice de caractéristiques
@app.get("/products", response_model=List[ProductResponse], tags=["Catalogue"])
def get_all_products():
    if df.empty:
        raise HTTPException(status_code=500, detail="Le jeu de données n'est pas chargé.")
    return df.to_dict(orient="records")

In [ ]:
# Chemin pour obtenir des recommandations de produits en fonction des identifiants sélectionnés et des ingrédients exclus.
@app.post("/recommend", response_model=List[ProductResponse], tags=["Recommandation"])
def get_recommendations(request: RecommendationRequest):
    if df.empty:
        raise HTTPException(status_code=500, detail="Le jeu de données n'est pas chargé.")
    indices = df.index[df["id"].isin(request.selected_ids)].tolist()
    if not indices:
        raise HTTPException(status_code=404, detail="Aucun produit valide trouvé pour les identifiants sélectionnés.")

    results = df.copy()
    results["similarity_score"] = np.round(np.mean(cosine_sim[indices], axis=0), 3)
    results = results[~results["id"].isin(request.selected_ids)]
    for ingredient in request.excluded_ingredients:
        ingredient = str(ingredient).strip()
        if ingredient:
            excluded_ids = ingredient_data.loc[ingredient_data["label_name"].str.contains(ingredient, case=False, na=False, regex=False), "product_id"].unique()
            results = results[~results["id"].isin(excluded_ids)]
    return results.sort_values("similarity_score", ascending=False).head(request.top_n).to_dict(orient="records")


In [7]:
# Chargement des données et calcul des similarités.
df, ingredient_data = load_and_prepare_data(CSV_PATH, PRODUCT_INGREDIENTS_PATH)
feature_matrix = feature_vectorizer.fit_transform(build_feature_rows(df, ingredient_data))
cosine_sim = cosine_similarity(feature_matrix, feature_matrix)
print(f"{len(df)} produits chargés et {feature_matrix.shape[1]} caractéristiques binaires.")


1000 produits chargés et 3076 caractéristiques binaires.


## III - Démarrer le site en local

Exécutez cette cellule pour lancer le site à `http://127.0.0.1:8000/`.


In [ ]:
# Démarrage du serveur local
server = uvicorn.Server(uvicorn.Config(app, host="127.0.0.1", port=8000, log_level="info"))
server_thread = threading.Thread(target=server.run, daemon=True)
server_thread.start()
for _ in range(100):
    if server.started:
        break
    if not server_thread.is_alive():
        raise RuntimeError("Le serveur web s'est arrêté avant d'avoir fini de démarrer.")
    time.sleep(0.1)
if not server.started:
    raise TimeoutError("Le serveur web n'a pas démarré dans les 10 secondes.")
print("Le site est accessible à l'adresse http://127.0.0.1:8000/")


INFO:     Started server process [20976]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)


Le site est accessible à l'adresse http://127.0.0.1:8000/


INFO:     127.0.0.1:57675 - "GET / HTTP/1.1" 200 OK
INFO:     127.0.0.1:57675 - "GET /products HTTP/1.1" 200 OK
INFO:     127.0.0.1:56820 - "POST /recommend HTTP/1.1" 200 OK
INFO:     127.0.0.1:56820 - "GET /images/17285.jpeg HTTP/1.1" 200 OK
INFO:     127.0.0.1:64402 - "GET /images/157759.jpeg HTTP/1.1" 200 OK
INFO:     127.0.0.1:64792 - "GET /images/133486.jpeg HTTP/1.1" 200 OK
INFO:     127.0.0.1:51328 - "GET /images/160693.jpeg HTTP/1.1" 200 OK
INFO:     127.0.0.1:62402 - "GET /images/177551.jpeg HTTP/1.1" 200 OK
INFO:     127.0.0.1:54891 - "GET /images/169169.jpeg HTTP/1.1" 200 OK
INFO:     127.0.0.1:51328 - "GET /images/100131.jpeg HTTP/1.1" 200 OK
INFO:     127.0.0.1:64792 - "GET /images/74506.jpeg HTTP/1.1" 200 OK
INFO:     127.0.0.1:54891 - "GET /images/129947.jpeg HTTP/1.1" 200 OK
INFO:     127.0.0.1:62402 - "GET /images/130991.jpeg HTTP/1.1" 200 OK
INFO:     127.0.0.1:56820 - "GET /images/78052.jpeg HTTP/1.1" 200 OK


## IV - Arrêter le serveur local

Exécutez cette cellule pour arrêter proprement le serveur.


In [ ]:
# Arrêt propre du serveur local
if server_thread and server_thread.is_alive():
    server.should_exit = True
    server_thread.join(timeout=10)
    if server_thread.is_alive():
        raise TimeoutError("Le serveur web ne s'est pas arrêté dans les 10 secondes.")
    print("Le serveur du site est arrêté.")
else:
    print("Le serveur du site n'est pas en cours d'exécution.")


INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [20976]


Le serveur du site est arrêté.
